# find more cPeaks

In [2]:
import numpy as np

data= np.array(open('./data/cpeaks_observed.bed').readlines())
len_ = np.array([int(i.split('\t')[2])-int(i.split('\t')[1]) for i in data])
data = list(data[len_>=100])
data = [[i.split('\t')[0],int(i.split('\t')[1]),int(i.split('\t')[2])] for i in data]

data = [i if i[2]-i[1] < 2000 else [i[0],i[1],i[1]+2000] for i in data]
with open('./data/cpeaks_observed_filtered.bed','w') as f:
    f.write('\n'.join(['\t'.join([i[0],str(i[1]),str(i[2])]) for i in data]))


In [3]:
import os

cmd1 = "cat ./data/hg38_blacklist_v2.bed ./data/cpeaks_observed.bed > ./data/combined_excl.bed"
cmd2 = "bedtools shuffle -i ./data/cpeaks_observed_filtered.bed -g ./data/hg38.chrom24.sizes.txt -excl ./data/combined_excl.bed -noOverlapping -maxTries 10 > ./data/negatives.bed"
cmd3 = "bedtools nuc -fi ./data/hg38.fa -bed ./data/cpeaks_observed_filtered.bed > ./data/pos_nucleotide_composition.txt"
cmd4 = "bedtools nuc -fi ./data/hg38.fa -bed ./data/negatives.bed > ./data/neg_nucleotide_composition.txt"

os.system(cmd1)
os.system(cmd2)
os.system(cmd3)
os.system(cmd4)

0

In [4]:
import pandas as pd

sub1 = './data/pos_nucleotide_composition.txt'
sub2 = './data/neg_nucleotide_composition.txt'

pos = pd.read_table(sub1)
neg = pd.read_table(sub2)

tmp1 = pos['10_num_N']
tmp2 = neg['10_num_N']

pos2 = pos[tmp1==0].copy()
neg2 = neg[tmp2==0].copy()

neg2.to_csv('./data/neg_filterN.bed',sep='\t',index = 0,columns=None)
pos2.to_csv('./data/pos_filterN.bed',sep='\t',index = 0,columns=None)

In [5]:
cmd1 = 'bedtools getfasta -fi ./data/hg38.fa -bed ./data/neg_filterN.bed -fo ./data/neg_filterN.fa'
cmd2 = 'bedtools getfasta -fi ./data/hg38.fa -bed ./data/pos_filterN.bed -fo ./data/pos_filterN.fa'

os.system(cmd1)
os.system(cmd2)

0

In [6]:
import pandas as pd
from pybedtools import BedTool

bed_file = pd.read_csv('./data/combined_excl.bed', sep='\t', header=None, names=['chr', 'start', 'end'])
chrom_sizes = pd.read_csv('./data/hg38.chrom24.sizes.txt', sep='\t', header=None, names=['chr', 'size'])
chr_list = ['chr' + str(i) for i in range(1, 23)]

ref_bed_file = chrom_sizes[chrom_sizes['chr'].isin(chr_list)]
ref_bed_file['start'] = 0
ref_bed_file['end'] = ref_bed_file['size'].copy()
ref_bed_file = ref_bed_file.drop('size',axis=1)

bed_gr = BedTool.from_dataframe(bed_file)
ref_gr = BedTool.from_dataframe(ref_bed_file)

missing_regions = ref_gr.subtract(bed_gr)

missing_regions.saveas('data/bg.bed')

/tmp/ipykernel_175158/285762841.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ref_bed_file['start'] = 0
/tmp/ipykernel_175158/285762841.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ref_bed_file['end'] = ref_bed_file['size'].copy()


<BedTool(data/bg.bed)>

In [7]:
with open('data/bg.bed', 'r') as f:
    lines = f.readlines()

win = 500
step = 250

with open('data/candidate.bed', 'w') as f:
    for line in lines:
        chrom, start, end, *rest = line.split('\t')
        start, end = int(start), int(end)
        
        length = end - start
        if length < 100:
            continue
        if length < 499:
            f.write(f"{chrom}\t{start}\t{end}\n")
            continue
        for i in range(start, end - win, step):
            f.write(f"{chrom}\t{i}\t{i+win}\n")
            
!bedtools getfasta -fi ./data/hg38.fa -bed ./data/candidate.bed -fo ./data/candidate.fa

# cluster the shape

In [1]:
data = open('./data/encode_shape.bed').readlines()
data2 = [i for i in data if int(i.strip().split('\t')[-1]) < 170 or int(i.strip().split('\t')[-2]) < 170]
data3 = [i for i in data if int(i.strip().split('\t')[-1]) > 170 and int(i.strip().split('\t')[-2]) > 170]

label_left = 0
label_right = 0

for d in data2:
    tmp = d.strip().split()
    if int(tmp[3]) < 170:
        label_left += 1
    
    
    if int(tmp[4]) < 170:
        label_right += 1

label0 = label_left+label_right
label1 = len(data2)*2 - label0

In [4]:
import random
data4 = random.sample(data3, (label0-label1)//2)
with open('./data/encode_shape_sub.bed','w') as f:
    f.write(''.join(data2 + data4))

In [5]:
import os

cmd1 = 'bedtools getfasta -fi ./data/hg38.fa -bed ./data/encode_shape_sub.bed -fo ./data/encode_shape_sub.fa'

os.system(cmd1)

0

In [2]:
cmd2 = 'bedtools getfasta -fi ./data/hg38.fa -bed ./data/cpeak_unlabeled.bed -fo ./data/cpeak_unlabeled.fa'

os.system(cmd2)
print('done')

done
